# 文档检索中的上下文压缩

## 概述

该代码演示了使用 LangChain 和 OpenAI 语言模型在文档检索系统中实现上下文压缩。该技术旨在通过压缩和提取给定查询上下文中文档最相关的部分来提高检索信息的相关性和简洁性。

## 动机

传统的文档检索系统通常返回整个块或文档，其中可能包含不相关的信息。上下文压缩通过智能地提取和压缩检索到的文档中最相关的部分来解决这个问题，从而实现更有针对性和更高效的信息检索。

## 关键组件

1. 支持从PDF文档创建存储
2. 基础搜索器设置
3. 基于LLM的上下文压缩器
4. 上下文压缩搜索器
5. 问答链集成压缩搜索器

## 方法详细信息

### 文档预处理和支持存储创建

1. 使用自定义 `encode_pdf` 函数对 PDF 进行处理并编码到支持存储中。

### 搜索器和压缩器设置

1. 从支持存储创建基础搜索器。
2. 使用 OpenAI 的 GPT-4 模型初始化基于 LLM 的上下文压缩器 (LLMChainExtractor)。

### 上下文压缩搜索器

1. 基础搜索器和压缩器合并为 ContextualCompression 搜索器。
2. 该检索器首先使用基本检索器获取文档，然后应用压缩器来提取最相关的信息。

### 问答链

1. 创建RetrievalQA链，集成压缩检索器。
2. 该链使用压缩和提取的信息来生成查询答案。

## 这种方法的好处

1. 提高相关性：系统只返回与查询最相关的信息。
2. 提高效率：通过压缩和提取相关部分，减少了LLM需要处理的文本量。
3. 增强上下文理解：基于LLM的压缩器可以理解查询的上下文并相应地提取信息。
4. 灵活性：系统可以轻松适应不同类型的文档和查询。

## 结论

文档检索中的上下文压缩提供了一种增强信息检索系统的质量和效率的强大方法。通过智能地提取和压缩相关信息，它可以为查询提供更有针对性和上下文感知的响应。这种方法在需要从大型文档集合中高效、准确地检索信息的各个领域都有潜在的应用。

<div style="text-align: center;">

<img src="../images/contextual_compression.svg" alt="contextual compression" style="width:70%; height:auto;">
</div>

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install langchain python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [ ]:
import os
import sys
from dotenv import load_dotenv
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers import ContextualCompressionRetriever
from langchain.chains import RetrievalQA


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### 定义文档的路径

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [2]:
path = "data/Understanding_Climate_Change.pdf"

### 创建一个支持存储

In [3]:
vector_store = encode_pdf(path)

### 创建搜索器 + 上下文压缩器 + 将它们组合起来

In [7]:
# Create a retriever
retriever = vector_store.as_retriever()


#Create a contextual compressor
llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini", max_tokens=4000)
compressor = LLMChainExtractor.from_llm(llm)

#Combine the retriever with the compressor
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

# Create a QA chain with the compressed retriever
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=compression_retriever,
    return_source_documents=True
)

### 用法示例

In [6]:
query = "What is the main topic of the document?"
result = qa_chain.invoke({"query": query})
print(result["result"])
print("Source documents:", result["source_documents"])

The main topic of the document is climate change, focusing on international collaboration, national strategies, policy development, and the ethical dimensions of climate justice. It discusses frameworks like the UNFCCC and the Paris Agreement, as well as the importance of sustainable practices for future generations.
Source documents: [Document(metadata={'source': '../data/Understanding_Climate_Change.pdf', 'page': 9}, page_content='Chapter 6: Global and Local Climate Action  \nInternational Collaboration  \nUnited Nations Framework Convention on Climate Change (UNFCCC)  \nThe UNFCCC is an international treaty aimed at addressing climate change. It provides a \nframework for negotiating specific protocols and agreements, such as the Kyoto Protocol and \nthe Paris Agreement. Global cooperation under the UNFCCC is crucial for coordinated \nclimate action.  \nParis Agreement  \nThe Paris Agreement, adopted in 2015, aims to limit global warming to well below 2 degrees \nCelsius above pre-i

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--contextual-compression)